# Phase 3 input builder: hard viscosity ($\nu = 0.01/\pi$)

**WISER x BQP 2026, Track A | Burgers at the Shah/Raissi viscosity**

This notebook produces **every input** needed for Workstreams A, B and C on the hard-viscosity
corpus, in the same schemas used at $\nu=0.05$, so the two viscosities can be concatenated in
Phase 4 without reconciliation work.

## Why one notebook rather than three

At $\nu=0.05$ the three workstreams were extracted in separate passes, each rebuilding the same
models. The expensive step is loading and evaluating a model, not measuring it, so here every
model is rebuilt **once** and all measurements are taken from that single pass.

## What this fixes, learned from the nu=0.05 round

| Problem then | Fix here |
|---|---|
| No QAPINN gradient-variance file at hard nu, so WS-B has no headline | Computed here (cell 5), 25 fresh initialisations per width |
| Gradient files recorded no model and no PDE, forcing inference | `model`, `pde`, `nu`, `n_feat` written as explicit columns |
| `wall_s` never reached disk, so cost could not be reported | A timing probe measures cost per step directly (cell 6) |
| Activations stored at one or two slices only | Re-extracted at **26 slices** spanning $t\in[0,1]$ |
| Layer-to-layer CKA never computed, so it had to be deferred | Computed here at every slice |
| Files silently dropped by a narrow tag filter | Every checkpoint accounted for, with an assertion that counts close |

## The training-budget asymmetry, and how it is handled

Read from the stored histories: the twin and GAAF ran **2016 steps at every width**, while the
QAPINN ran **4400 at q3-q7** and **1000 at q8**. Two consequences are recorded in every table:

- `budget_steps` is written per run, so no analysis can silently compare unequal budgets.
- `in_primary` marks the analysis set. **q8 is excluded from the primary set**, because its
  budget differs from both the other QAPINN widths and the classical baselines. q8 is retained in
  every file and reported separately.

The QAPINN still trained 2.2x longer than the classical baselines at q3-q7. That is not
removable, so the WS-A analysis below computes a **recovery fraction against the analytical
solution** as its primary metric, which compares each model to ground truth rather than to the
other models and is therefore unaffected by how long either trained.

## Scope note

No ablation grid (entanglement topology, output extraction) was run at this viscosity. That
finding stays scoped to $\nu=0.05$ and is not reproduced here.

## Output

Everything is written to **`WISER Results/Phase 3 Hard/`**.

## 1. Mount and set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, glob, re, json, csv, math, time, shutil
from collections import Counter, defaultdict
import numpy as np

MYDRIVE = "/content/drive/MyDrive"
ROOT = None
for c in sorted(glob.glob(os.path.join(MYDRIVE, "**", "WISER Results"), recursive=True)):
    if os.path.isdir(c): ROOT = c; break
assert ROOT, "WISER Results not found"
OUT = os.path.join(ROOT, "Phase 3 Hard")
for sub in ["", "fields", "spectra", "figures"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print("WISER Results :", ROOT)
print("output folder :", OUT)

NU_HARD = 0.01/np.pi
N_SLICES = 26
T_SLICES = np.round(np.linspace(0.0, 1.0, N_SLICES), 4)
KCUT = 8.0
N_GRID = 256
print("nu = %.6f | %d slices | k_cut = %g" % (NU_HARD, N_SLICES, KCUT))

NS = SRC_NB = None
ACCOUNT, REBUILT = [], {}

WISER Results : /content/drive/MyDrive/WISER Results
output folder : /content/drive/MyDrive/WISER Results/Phase 3 Hard
nu = 0.003183 | 26 slices | k_cut = 8


## 2. Import model classes

Executes only the class-definition cells of a hard-viscosity training notebook. Any cell that
*calls* the training machinery is refused, and execution stops as soon as the classes exist.

In [3]:
def is_driver(src):
    for name in ("run_all", "run_one", "run_sweep", "train_schedule", "fit", "main"):
        if (name + "(") in src and ("def " + name) not in src: return True
    return False

def load_definitions(path, needed):
    nb = json.load(open(path)); g = {"__name__": "__main__"}; n = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code": continue
        s = "".join(c["source"])
        if is_driver(s): print("   [stop] driver cell reached, not executed"); break
        if "drive.mount" in s: continue
        clean = "\n".join(l for l in s.splitlines()
                          if not l.strip().startswith(("!", "%", "get_ipython")))
        try: exec(compile(clean, "<c%d>" % n, "exec"), g)
        except Exception as e: print("   [warn] cell %d: %s" % (n, type(e).__name__))
        n += 1
        if all(k in g for k in needed):
            print("   [ok] definitions complete after %d cells" % n); break
    return g, n

cands = [p for p in glob.glob(os.path.join(MYDRIVE, "**", "*.ipynb"), recursive=True)
         if any(k in os.path.basename(p) for k in ("N4_GAAF", "N3_TWIN", "N1_QAPINN",
                                                   "N2q", "GAAF", "PHASE2_FINAL", "heat"))]
print("candidates: %d" % len(cands))
PREF = ["QAPINN", "ClassicalTwin", "GAAFPINN", "TWIN_CONFIG"]
ESS  = ["QAPINN", "ClassicalTwin", "TWIN_CONFIG"]
order = sorted(cands, key=lambda q: (("GAAF" not in q), len(q)))
for label, need in (("pass 1 (all four)", PREF), ("pass 2 (essential)", ESS)):
    print("\n--- %s ---" % label)
    for p in order:
        print("trying", os.path.basename(p))
        try: g, n = load_definitions(p, need)
        except Exception as e: print("   failed:", type(e).__name__); continue
        print("   found:", [k for k in PREF if k in g])
        if all(k in g for k in need):
            NS, SRC_NB = g, p; print("   [OK] using", os.path.basename(p)); break
    if NS: break
assert NS is not None, "could not import model classes"

import torch, torch.nn as nn
QAPINN = NS["QAPINN"]; ClassicalTwin = NS["ClassicalTwin"]; GAAFPINN = NS.get("GAAFPINN")
device = NS.get("device", torch.device("cpu")); DTYPE = NS.get("DTYPE", torch.float32)
AVAIL = {"qapinn", "twin"} | ({"gaaf"} if GAAFPINN is not None else set())
print("\nrebuildable kinds:", sorted(AVAIL))
if GAAFPINN is None:
    print("[WARNING] GAAFPINN unavailable; GAAF runs will be reported as unavailable.")

def build(kind, n):
    if kind == "qapinn":
        return QAPINN(n_qubits=n, n_layers=6, entanglement="all",
                      measurement="expval", head_width=20, head_depth=5)
    if kind == "twin": return ClassicalTwin(n_feat=n, head_width=20, head_depth=5)
    if kind == "gaaf" and GAAFPINN is not None:
        return GAAFPINN(n_feat=n, head_width=20, head_depth=5)
    return None

candidates: 8

--- pass 1 (all four) ---
trying N4_GAAF_hardnu_q3to8.ipynb
[OK] Phase 3 directory: quapinns/phase3_hardnu_gaaf
[ok] torch
[ok] pennylane
[ok] scipy
[ok] numpy
[ok] matplotlib
[done]
[cfg] torch 2.11.0+cpu | pennylane 0.45.1 | cpu | torch.float32
[cfg] outputs -> qapinn_runs_hardnu_gaaf/  | quantum sim: default.qubit (backprop) on cpu
[hw] host=e5556e80c4d2 | platform=Linux-6.6.122+-x86_64-with-glibc2.35 | processor=x86_64 | python=3.12.13 | torch=2.11.0+cpu | pennylane=0.45.1 | device=cpu
[hw] NOTE: wall-clock timings are only comparable WITHIN one machine.
[ok] data maker ready
[ok] references ready
[ok] QAPINN ready
[ok] loss ready
[ok] engine ready
[ok] recorder ready
[ok] ClassicalTwin ready
[ok] GAAF-PINN ready
   [ok] definitions complete after 11 cells
   found: ['QAPINN', 'ClassicalTwin', 'GAAFPINN', 'TWIN_CONFIG']
   [OK] using N4_GAAF_hardnu_q3to8.ipynb

rebuildable kinds: ['gaaf', 'qapinn', 'twin']


## 3. Analytical reference and metric definitions

In [4]:
def burgers_exact(x, ts, nu=NU_HARD, n_quad=8000):
    x = np.asarray(x, float); ts = np.atleast_1d(np.asarray(ts, float))
    U = np.zeros((len(ts), len(x)))
    y = np.linspace(-3.0, 3.0, n_quad)
    cos_t = np.cos(np.pi*y)/(2.0*np.pi*nu)
    for i, t in enumerate(ts):
        if t < 1e-12: U[i] = -np.sin(np.pi*x); continue
        F = cos_t.reshape(1, -1) + (x.reshape(-1, 1) - y.reshape(1, -1))**2/(4.0*nu*t)
        ml = np.max(-F, axis=1, keepdims=True); w = np.exp(-F - ml)
        num = np.sum((x.reshape(-1, 1) - y.reshape(1, -1))/t * w, axis=1)
        den = np.sum(w, axis=1)
        U[i] = np.where(den > 0, num/den, 0.0)
    return U

XS = np.linspace(-1, 1, N_GRID)
FREQS = np.fft.rfftfreq(N_GRID, d=(2.0/N_GRID))
U_EXACT = burgers_exact(XS, T_SLICES)
print("analytical reference computed:", U_EXACT.shape)

def spectrum(u):
    return np.abs(np.fft.rfft(u - u.mean()))**2

def centroid(P):
    tot = P.sum(); return float((FREQS*P).sum()/tot) if tot > 0 else 0.0

def hf_frac(P, kcut=KCUT):
    tot = P.sum(); return float(P[FREQS > kcut].sum()/tot) if tot > 0 else 0.0

P_EXACT = np.array([spectrum(U_EXACT[i]) for i in range(len(T_SLICES))])
CEN_EXACT = np.array([centroid(P_EXACT[i]) for i in range(len(T_SLICES))])
PHI_EXACT = np.array([hf_frac(P_EXACT[i]) for i in range(len(T_SLICES))])
print("analytical phi_>8 : min %.3e  max %.3e at t=%.2f"
      % (PHI_EXACT.min(), PHI_EXACT.max(), T_SLICES[int(np.argmax(PHI_EXACT))]))
print("this is the signal that was ABSENT at nu=0.05 (phi ~ 1e-6 there)")

def linear_cka(X, Y):
    X = np.asarray(X, float); Y = np.asarray(Y, float)
    if X.ndim == 1: X = X.reshape(-1, 1)
    if Y.ndim == 1: Y = Y.reshape(-1, 1)
    X = X - X.mean(0, keepdims=True); Y = Y - Y.mean(0, keepdims=True)
    num = np.linalg.norm(Y.T @ X, "fro")**2
    den = np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")
    return float(num/den) if den > 0 else 0.0

_r = np.random.default_rng(0); _X = _r.normal(size=(64, 8))
_Q, _ = np.linalg.qr(_r.normal(size=(8, 8)))
assert abs(linear_cka(_X, _X)-1) < 1e-9 and abs(linear_cka(_X, _X@_Q)-1) < 1e-9
assert abs(linear_cka(_X, 5.5*_X)-1) < 1e-9 and linear_cka(_X, np.ones((64, 3))) == 0.0
print("[ok] linear CKA verified")

analytical reference computed: (26, 256)
analytical phi_>8 : min 1.398e-06  max 1.875e-02 at t=0.68
this is the signal that was ABSENT at nu=0.05 (phi ~ 1e-6 there)
[ok] linear CKA verified


## 4. Inventory and accounting

Records the iteration budget and host of every run, so both known confounds are visible before
any analysis.

In [5]:
RE = re.compile(r"MS_hard_(?P<pde>burgers)_n(?P<n>\d+)_(?P<model>qapinn|twin|gaaf)"
               r"_s(?P<seed>\d+)(?P<retry>_r\d+)?")

def parse(tag):
    m = RE.search(tag)
    if not m: return None
    d = m.groupdict()
    return dict(pde=d["pde"], n_feat=int(d["n"]), model=d["model"],
                seed=int(d["seed"]), retry=bool(d["retry"]))

CK = glob.glob(os.path.join(MYDRIVE, "**", "checkpoints", "MS_hard_*.pt"), recursive=True)
print("hard-nu checkpoints found: %d" % len(CK))

HOST = {}
for p in glob.glob(os.path.join(MYDRIVE, "**", "hardware.json"), recursive=True):
    try: HOST[os.path.dirname(os.path.dirname(p))] = json.load(open(p)).get("host")
    except Exception: pass

seen, TARGETS = {}, []
for p in sorted(CK):
    tag = os.path.basename(p)[:-3]; rel = p.replace(MYDRIVE, "").lstrip("/")
    m = parse(tag)
    if m is None:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="tag not recognised")); continue
    if tag in seen:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="duplicate tag, kept %s" % seen[tag])); continue
    seen[tag] = rel; TARGETS.append((tag, p, m))
    ACCOUNT.append(dict(tag=tag, path=rel, decision="selected", reason=""))

sel = [a for a in ACCOUNT if a["decision"] == "selected"]
exc = [a for a in ACCOUNT if a["decision"] == "excluded"]
print("  selected %d | excluded %d" % (len(sel), len(exc)))
for r, c in Counter(a["reason"] for a in exc).most_common(): print("     %4d %s" % (c, r))
assert len(sel)+len(exc) == len(CK), "ACCOUNTING FAILURE"
print("[OK] accounting closes: %d + %d = %d" % (len(sel), len(exc), len(CK)))

cov = defaultdict(list)
for tag, _, m in TARGETS:
    cov[(m["model"], m["n_feat"])].append(("%dr" % m["seed"]) if m["retry"] else str(m["seed"]))
print("\ncoverage:")
for k in sorted(cov): print("  %-7s n=%d: %s" % (k[0], k[1], sorted(cov[k])))

hard-nu checkpoints found: 116
  selected 58 | excluded 58
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n3_gaaf_s1234.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n3_gaaf_s1234_r1.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n3_gaaf_s2025.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n3_gaaf_s777.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n4_gaaf_s1234.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints/MS_hard_burgers_n4_gaaf_s1234_r1.pt
        1 duplicate tag, kept WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapi

## 5. QAPINN gradient variance at hard viscosity

This did not exist. Without it Workstream B has no matched comparison at this viscosity. Same
estimator as the $\nu=0.05$ round: 25 fresh initialisations, the physics-informed loss
differentiated once, variance over the first-layer gradients only.

In [6]:
make_data = NS.get("make_training_data"); pinn_loss = NS.get("pinn_loss")
assert make_data is not None and pinn_loss is not None, \
    "make_training_data / pinn_loss not found in the source notebook"

def grad_variance(kind, n_feat, n_samples=25, n_f=400, seed=1234):
    x_u, t_u, u_u, x_f, t_f = make_data("burgers", n_f, seed=seed)
    comps = []
    for s in range(n_samples):
        torch.manual_seed(seed + 1000*s)
        m = build(kind, n_feat)
        if m is None: return float("nan")
        m = m.to(device)
        if kind == "qapinn":
            with torch.no_grad():
                for p_ in m.qlayer.parameters(): p_.uniform_(0, 2*math.pi)
            target = m.qlayer
        else:
            target = m.front
        m.zero_grad()
        loss, _, _ = pinn_loss(m, x_u, t_u, u_u, x_f, t_f, pde="burgers", nu=NU_HARD)
        loss.backward()
        g = torch.cat([p_.grad.flatten() for p_ in target.parameters() if p_.grad is not None])
        comps.append(g.detach().cpu().numpy())
    return float(np.var(np.concatenate(comps)))

GRAD = []
widths = sorted({m["n_feat"] for _, _, m in TARGETS})
print("computing gradient variance for widths", widths)
for kind in ["qapinn", "twin", "gaaf"]:
    if kind not in AVAIL:
        print("  [skip] %s not rebuildable" % kind); continue
    for n in widths:
        t0 = time.perf_counter()
        v = grad_variance(kind, n)
        GRAD.append(dict(model=kind, pde="burgers", nu="0.01/pi", n_feat=n, variance=v,
                         n_samples=25, n_collocation=400, source="computed_here"))
        print("  %-7s n=%d : var = %.6e   (%.1f s)" % (kind, n, v, time.perf_counter()-t0))

print("\nfitted decay (log-linear), all widths:")
for kind in sorted({g["model"] for g in GRAD}):
    xs = np.array([g["n_feat"] for g in GRAD if g["model"] == kind], float)
    ys = np.array([g["variance"] for g in GRAD if g["model"] == kind], float)
    ok = np.isfinite(ys) & (ys > 0)
    if ok.sum() >= 3:
        b, a = np.polyfit(xs[ok], np.log(ys[ok]), 1)
        r2 = 1 - np.sum((np.log(ys[ok])-(a+b*xs[ok]))**2)/np.sum((np.log(ys[ok])-np.log(ys[ok]).mean())**2)
        print("  %-7s slope %+.4f /qubit  R2 %.3f  total decay %.1fx"
              % (kind, b, r2, ys[ok][0]/ys[ok][-1]))

computing gradient variance for widths [3, 4, 5, 6, 7, 8]
  qapinn  n=3 : var = 4.895546e-01   (10.2 s)
  qapinn  n=4 : var = 2.093788e-01   (16.4 s)
  qapinn  n=5 : var = 1.120753e-01   (27.1 s)
  qapinn  n=6 : var = 1.291275e-01   (53.4 s)
  qapinn  n=7 : var = 1.375910e-01   (105.6 s)
  qapinn  n=8 : var = 3.532573e-02   (231.7 s)
  twin    n=3 : var = 2.909451e+00   (0.4 s)
  twin    n=4 : var = 7.410987e+00   (0.4 s)
  twin    n=5 : var = 1.197512e+00   (0.4 s)
  twin    n=6 : var = 1.690527e+00   (0.5 s)
  twin    n=7 : var = 2.757738e+00   (0.5 s)
  twin    n=8 : var = 2.838107e+00   (0.4 s)
  gaaf    n=3 : var = 1.523847e+03   (0.4 s)
  gaaf    n=4 : var = 3.058080e+03   (0.3 s)
  gaaf    n=5 : var = 8.565810e+02   (0.3 s)
  gaaf    n=6 : var = 3.977238e+02   (0.4 s)
  gaaf    n=7 : var = 7.491310e+02   (0.3 s)
  gaaf    n=8 : var = 8.588088e+02   (0.3 s)

fitted decay (log-linear), all widths:
  gaaf    slope -0.2244 /qubit  R2 0.366  total decay 1.8x
  qapinn  slope -0.4075 /

## 6. Timing probe

The training engine writes elapsed time only after the final checkpoint, so no stored run carries
it. Cost per step is measured directly here, on this machine, so the numbers are internally
comparable even though they cannot be compared with runs made elsewhere.

In [7]:
# Memory-safe timing probe. The earlier version used 2000 collocation points with no
# chunking, which needs ~8 GB at n=8 and crashes the kernel. Cost per step is linear
# in the collocation count, so 400 points preserves the scaling with width, which is
# the quantity of interest. The absolute number is reported at 400 points, not 2000.
import gc
make_data = NS.get("make_training_data"); pinn_loss = NS.get("pinn_loss")
assert make_data is not None and pinn_loss is not None, "re-run cell 2"

N_F_PROBE, K = 400, 5
TIMING = []
x_u, t_u, u_u, x_f, t_f = make_data("burgers", N_F_PROBE, seed=1234)
for kind in ["qapinn", "twin", "gaaf"]:
    if kind not in AVAIL: continue
    for n in widths:
        try:
            m = build(kind, n)
            if m is None: continue
            m = m.to(device)
            opt = torch.optim.Adam(m.parameters(), lr=1e-3)
            for _ in range(2):                      # warm-up, not timed
                opt.zero_grad()
                l, _, _ = pinn_loss(m, x_u, t_u, u_u, x_f, t_f, pde="burgers", nu=NU_HARD)
                l.backward(); opt.step()
            t0 = time.perf_counter()
            for _ in range(K):
                opt.zero_grad()
                l, _, _ = pinn_loss(m, x_u, t_u, u_u, x_f, t_f, pde="burgers", nu=NU_HARD)
                l.backward(); opt.step()
            dt = (time.perf_counter() - t0) / K
            TIMING.append(dict(model=kind, n_feat=n, sec_per_step=dt,
                               n_collocation=N_F_PROBE, measured_on="this machine"))
            print("  %-7s n=%d : %.3f s/step  (at %d collocation points)"
                  % (kind, n, dt, N_F_PROBE))
        except Exception as e:
            print("  %-7s n=%d : FAILED %s" % (kind, n, type(e).__name__))
            TIMING.append(dict(model=kind, n_feat=n, sec_per_step=float("nan"),
                               n_collocation=N_F_PROBE, measured_on="failed"))
        finally:
            for v in ("m", "opt", "l"):
                if v in dir(): pass
            try: del m, opt
            except Exception: pass
            gc.collect()
print("\ntiming rows: %d" % len(TIMING))

  qapinn  n=3 : 0.313 s/step  (at 400 collocation points)
  qapinn  n=4 : 0.582 s/step  (at 400 collocation points)
  qapinn  n=5 : 0.994 s/step  (at 400 collocation points)
  qapinn  n=6 : 2.070 s/step  (at 400 collocation points)
  qapinn  n=7 : 4.420 s/step  (at 400 collocation points)
  qapinn  n=8 : 8.803 s/step  (at 400 collocation points)
  twin    n=3 : 0.018 s/step  (at 400 collocation points)
  twin    n=4 : 0.009 s/step  (at 400 collocation points)
  twin    n=5 : 0.009 s/step  (at 400 collocation points)
  twin    n=6 : 0.009 s/step  (at 400 collocation points)
  twin    n=7 : 0.009 s/step  (at 400 collocation points)
  twin    n=8 : 0.009 s/step  (at 400 collocation points)
  gaaf    n=3 : 0.011 s/step  (at 400 collocation points)
  gaaf    n=4 : 0.014 s/step  (at 400 collocation points)
  gaaf    n=5 : 0.011 s/step  (at 400 collocation points)
  gaaf    n=6 : 0.013 s/step  (at 400 collocation points)
  gaaf    n=7 : 0.012 s/step  (at 400 collocation points)
  gaaf    n=8 

## 7. The single rebuild pass

Each model is loaded once. From that one load we take the loss history, the spectra and fields at
26 slices, the activations, both CKA families and the per-neuron statistics.

In [ ]:
@torch.no_grad()
def predict(model, ts, chunk=1024):
    out = np.empty((len(ts), N_GRID))
    for i, t in enumerate(ts):
        xin = torch.tensor(XS.reshape(-1, 1), dtype=DTYPE, device=device)
        tin = torch.full_like(xin, float(t))
        out[i] = model(xin, tin).cpu().numpy().ravel()
    return out

@torch.no_grad()
def acts_at(model, t):
    xin = torch.tensor(XS.reshape(-1, 1), dtype=DTYPE, device=device)
    tin = torch.full_like(xin, float(t))
    store, hooks = {}, []
    first = getattr(model, "qlayer", None); fname = "L0_quantum"
    if first is None: first, fname = getattr(model, "front", None), "L0_front"
    if first is not None:
        hooks.append(first.register_forward_hook(
            lambda m, i, o, k=fname: store.__setitem__(k, o.detach().cpu().numpy())))
    k = 0
    for mod in model.head:
        if isinstance(mod, nn.Tanh) or mod.__class__.__name__ == "GAAFTanh":
            k += 1
            hooks.append(mod.register_forward_hook(
                lambda m, i, o, kk=k: store.__setitem__("L%d" % kk,
                                                        o.detach().cpu().numpy())))
    out = model(xin, tin).cpu().numpy()
    for h in hooks: h.remove()
    return store, out

WSA, WSB_HIST, WSC_CKA, WSC_NEU, RUNMETA, failures = [], [], [], [], [], []
for i, (tag, path, meta) in enumerate(TARGETS):
    if meta["model"] not in AVAIL:
        failures.append((tag, "class %s unavailable" % meta["model"])); continue
    try:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        model = build(meta["model"], meta["n_feat"])
        model.load_state_dict(ck["model_state"]); model.eval().to(device)
    except Exception as e:
        failures.append((tag, "rebuild %s" % type(e).__name__)); continue

    h = ck.get("history", {}) or {}
    loss = np.asarray(h.get("loss", []), float)
    budget = int(len(loss))
    host = None
    for k, v in HOST.items():
        if k.replace(MYDRIVE, "").lstrip("/") in path.replace(MYDRIVE, "").lstrip("/"):
            host = v; break
    # q8 leaves the primary set: its budget matches neither the other QAPINN widths
    # nor the classical baselines.
    in_primary = bool(meta["n_feat"] != 8)
    RUNMETA.append(dict(tag=tag, **meta, nu="0.01/pi", budget_steps=budget, host=host,
                        in_primary=in_primary, dead=bool(len(loss) > 1 and
                                                         loss.min() > 0.5*loss[0])))
    step = max(1, budget//400)
    for j in range(0, budget, step):
        WSB_HIST.append(dict(tag=tag, **meta, step=j, loss=float(loss[j]),
                             mse_u=float(h["mse_u"][j]) if j < len(h.get("mse_u", [])) else np.nan,
                             mse_f=float(h["mse_f"][j]) if j < len(h.get("mse_f", [])) else np.nan))

    U = predict(model, T_SLICES)
    err = U - U_EXACT
    l2 = float(np.linalg.norm(err)/np.linalg.norm(U_EXACT))
    Pm = np.array([spectrum(U[k]) for k in range(len(T_SLICES))])
    np.savez_compressed(os.path.join(OUT, "spectra", tag + "_spectrum.npz"),
                        x=XS, freqs=FREQS, power_model=Pm, power_exact=P_EXACT,
                        t_slices=T_SLICES, nu=NU_HARD, n_feat=meta["n_feat"],
                        kind=meta["model"], seed=meta["seed"])
    np.savez_compressed(os.path.join(OUT, "fields", tag + "_field.npz"),
                        x=XS, t=T_SLICES, u_pred=U.astype(np.float32),
                        u_exact=U_EXACT.astype(np.float32), err=err.astype(np.float32))

    for k, t in enumerate(T_SLICES):
        cm, pm = centroid(Pm[k]), hf_frac(Pm[k])
        WSA.append(dict(tag=tag, **meta, nu="0.01/pi", t=float(t), in_primary=in_primary,
                        budget_steps=budget,
                        centroid_model=cm, centroid_exact=float(CEN_EXACT[k]),
                        phi_model=pm, phi_exact=float(PHI_EXACT[k]),
                        # recovery fraction: share of the TRUE fine structure represented.
                        # Compares each model to ground truth, so it is unaffected by how
                        # long any model trained.
                        phi_recovery=float(pm/PHI_EXACT[k]) if PHI_EXACT[k] > 0 else np.nan,
                        l2_field=l2))

        A, out = acts_at(model, float(t))
        names = list(A)
        row = dict(tag=tag, **meta, nu="0.01/pi", t=float(t), in_primary=in_primary)
        for nm in names: row["cka_%s_out" % nm] = linear_cka(A[nm], out)
        for a in range(len(names)):          # layer-to-layer, the deferred item
            for b in range(a+1, len(names)):
                row["ckaLL_%s__%s" % (names[a], names[b])] = linear_cka(A[names[a]], A[names[b]])
        WSC_CKA.append(row)
        for nm in names:
            M = A[nm]
            for j in range(M.shape[1]):
                v = M[:, j]
                WSC_NEU.append(dict(tag=tag, **meta, nu="0.01/pi", t=float(t), layer=nm,
                                    neuron=j, mean=float(v.mean()), std=float(v.std()),
                                    sat=float(np.mean(np.abs(v) > 0.95)),
                                    cka_out=linear_cka(v, out)))
    REBUILT[tag] = True
    if (i+1) % 5 == 0: print("  %d/%d runs" % (i+1, len(TARGETS)))

print("\nrebuilt %d | failures %d" % (len(REBUILT), len(failures)))
for t, w in failures[:15]: print("   [fail] %s: %s" % (t, w))
print("WS-A rows %d | history rows %d | CKA rows %d | neuron rows %d"
      % (len(WSA), len(WSB_HIST), len(WSC_CKA), len(WSC_NEU)))

  5/58 runs
  10/58 runs
  15/58 runs
  20/58 runs
  25/58 runs
  30/58 runs
  35/58 runs


## 8. The high-frequency question

The measurement that was uninformative at $\nu=0.05$, because the target carried no
high-frequency content. Here it does. Three readings are reported, and they fail differently.

In [ ]:
P = [r for r in WSA if r["in_primary"]]
anchor = float(T_SLICES[int(np.argmax(PHI_EXACT))])
A = [r for r in P if abs(r["t"] - anchor) < 1e-9]
print("anchor slice t = %.2f, where the analytical phi_>8 peaks at %.4e\n"
      % (anchor, PHI_EXACT.max()))

print("=== (a) recovery fraction: share of TRUE fine structure represented ===")
print("    (primary metric: each model against ground truth, budget-independent)")
print("%5s %8s %10s %10s %10s" % ("n", "model", "mean", "sd", "runs"))
for n in sorted({r["n_feat"] for r in A}):
    for m in ["qapinn", "twin", "gaaf"]:
        v = [r["phi_recovery"] for r in A if r["n_feat"] == n and r["model"] == m]
        if v: print("%5d %8s %10.4f %10.4f %10d" % (n, m, np.mean(v), np.std(v), len(v)))

print("\n=== (b) head to head at matched width ===")
for n in sorted({r["n_feat"] for r in A}):
    q = [r["phi_model"] for r in A if r["n_feat"] == n and r["model"] == "qapinn"]
    t_ = [r["phi_model"] for r in A if r["n_feat"] == n and r["model"] == "twin"]
    if q and t_:
        print("  n=%d  QAPINN %.4e   twin %.4e   ratio %.3f"
              % (n, np.mean(q), np.mean(t_), np.mean(q)/max(np.mean(t_), 1e-30)))

print("\n=== (c) width trend, QAPINN (q8 excluded: budget differs) ===")
ns = sorted({r["n_feat"] for r in A if r["model"] == "qapinn"})
mu = [np.mean([r["phi_model"] for r in A if r["model"] == "qapinn" and r["n_feat"] == n])
      for n in ns]
print("  n =", ns); print("  phi =", ["%.4e" % v for v in mu])
if len(ns) >= 3:
    b = np.polyfit(ns, np.log(np.maximum(mu, 1e-30)), 1)[0]
    print("  log-linear slope %+.4f per qubit" % b)

print("\n=== (d) matched-accuracy pairing ===")
print("    Runs of comparable L2 compared on spectral content, which removes the")
print("    budget asymmetry entirely. Pairs within 25% relative L2.")
byrun = {}
for r in A: byrun[r["tag"]] = r
qs = [r for r in byrun.values() if r["model"] == "qapinn"]
ts = [r for r in byrun.values() if r["model"] == "twin"]
pairs = 0
for a in qs:
    for b in ts:
        if a["n_feat"] != b["n_feat"]: continue
        if abs(a["l2_field"]-b["l2_field"])/max(b["l2_field"], 1e-30) < 0.25:
            pairs += 1
            print("  n=%d  L2 %.3e vs %.3e   phi %.4e vs %.4e   ratio %.2f"
                  % (a["n_feat"], a["l2_field"], b["l2_field"], a["phi_model"],
                     b["phi_model"], a["phi_model"]/max(b["phi_model"], 1e-30)))
if pairs == 0:
    print("  no QAPINN/twin pair reached comparable accuracy at the same width;")
    print("  the matched-accuracy comparison is not available and must be reported as such.")

## 9. Write every table and the manifest

In [ ]:
def dump(rows, name):
    if not rows: print("[skip]", name); return
    p = os.path.join(OUT, name); keys = sorted({k for r in rows for k in r})
    with open(p, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
    print("[saved] %-34s %7d rows" % (name, len(rows)))

dump(WSA, "wsa_hard_metrics_by_slice.csv")
dump(GRAD, "wsb_hard_gradient_variance.csv")
dump(TIMING, "wsb_hard_timing.csv")
dump(WSB_HIST, "wsb_hard_loss_traces.csv")
dump(RUNMETA, "run_metadata.csv")
dump(WSC_CKA, "wsc_hard_cka_by_slice.csv")
dump(WSC_NEU, "wsc_hard_neuron_stats.csv")
dump(ACCOUNT, "checkpoint_accounting.csv")

man = dict(
    note="phase3-input-builder-hardnu (Track A, Djabon)",
    pde="burgers", nu=float(NU_HARD), n_slices=N_SLICES, k_cut=KCUT,
    anchor_slice=anchor, source_notebook=os.path.basename(SRC_NB),
    checkpoints_found=len(CK), selected=len(sel), excluded=len(exc),
    rebuilt=len(REBUILT), failures=[{"tag": a, "reason": b} for a, b in failures],
    analytical_phi_max=float(PHI_EXACT.max()),
    budgets={str(k): v for k, v in
             Counter((r["model"], r["n_feat"], r["budget_steps"]) for r in RUNMETA).items()},
    hosts=sorted({r["host"] for r in RUNMETA if r["host"]}),
    dead_runs=[r["tag"] for r in RUNMETA if r["dead"]],
    coverage={"%s|n%d" % (k[0], k[1]): sorted(v) for k, v in cov.items()},
    caveats=[
        "q8 is excluded from the primary analysis set (in_primary=False): its budget "
        "(~1000 steps) matches neither the other QAPINN widths (4400) nor the "
        "classical baselines (2016). It is retained in every file.",
        "At q3-q7 the QAPINN trained about 2.2x longer than the classical baselines. "
        "The recovery fraction (phi_model / phi_exact) is therefore the primary WS-A "
        "metric, since it compares each model to ground truth rather than to the others.",
        "Wall-clock from the stored runs is unavailable; the timing table was measured "
        "on this machine and is internally comparable only.",
        "No ablation grid (entanglement, output extraction) exists at this viscosity; "
        "that finding remains scoped to nu=0.05.",
        "Seed variance reflects initialisation only: training data is seeded globally.",
        "No value is interpolated or filled; missing configurations are reported missing.",
    ])
json.dump(man, open(os.path.join(OUT, "phase3_hard_manifest.json"), "w"), indent=2)
print("\n[saved] phase3_hard_manifest.json")

NB = "Phase3_Hard_InputBuilder.ipynb"
c2 = [q for q in glob.glob(os.path.join(MYDRIVE, "**", "*.ipynb"), recursive=True)
      if os.path.basename(q).startswith("Phase3_Hard_InputBuilder")]
if c2:
    src = max(c2, key=os.path.getmtime); dst = os.path.join(OUT, NB)
    if os.path.abspath(src) != os.path.abspath(dst): shutil.copy2(src, dst)
    print("[saved] notebook archived to", dst)
else:
    print("[note] notebook not found in Drive; File > Save a copy in Drive, then re-run.")

print("\nFinal contents of", OUT)
for f in sorted(os.listdir(OUT)):
    q = os.path.join(OUT, f)
    print("   %-40s %s" % (f, ("%d files" % len(os.listdir(q))) if os.path.isdir(q)
                           else "%d bytes" % os.path.getsize(q)))